# Агентный цикл генерации изображений (локально)

Адаптированная версия лабораторной работы под **локальные** серверы:

- **Генерация изображений**: `sd-server` (Krea-2) → `http://192.168.60.193:1235`
- **LLM + Vision**: LM Studio → `http://192.168.60.193:1234` (модель `qwen/qwen3.8-27b`)

Архитектура:

```
Концепция → LLM создаёт промпт → Krea-2 рисует → Vision LLM оценивает → улучшение промпта → ...
```


In [1]:
%pip install --upgrade openai pillow requests matplotlib -q

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
label-studio 1.22.0 requires numpy<3.0.0,>=2.2.6, but you have numpy 1.26.4 which is incompatible.
label-studio 1.22.0 requires openai<2.0.0,>=1.10.0, but you have openai 3.15.0 which is incompatible.
label-studio 1.22.0 requires pydantic>=2.9.2, but you have pydantic 1.10.26 which is incompatible.
label-studio 1.22.0 requires requests<2.33.0,>=2.32.3, but you have requests 2.34.2 which is incompatible.
tensorflow-intel 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Настройки серверов

Измените адреса, если у вас другие порты.

In [ ]:
import os
import io
import base64
import json
import requests
from IPython.display import Markdown, display
from PIL import Image
from openai import OpenAI

# ====================== НАСТРОЙКИ ======================
# LM Studio (LLM + Vision)
LMS_URL = "http://192.168.60.193:1234/v1"
LMS_MODEL = "qwen/qwen3.8-27b"          # точное имя модели из LM Studio

# sd-server (Krea-2)
SD_URL = "http://192.168.60.193:1235"   # порт вашего sd-server

# Параметры генерации Krea-2 Turbo
SD_STEPS = 8
SD_CFG = 1.0
SD_WIDTH = 256
SD_HEIGHT = 256
# =======================================================

def printx(string):
    display(Markdown(string))

print("✅ Настройки загружены")
print(f"   LLM/Vision : {LMS_URL}  ({LMS_MODEL})")
print(f"   Images     : {SD_URL}")

✅ Настройки загружены
   LLM/Vision : http://192.168.60.193:1234/v1  (qwen/qwen3.8-27b)
   Images     : http://192.168.60.193:1235


---
## Шаг 1: Инициализация клиентов

In [3]:
# OpenAI-совместимый клиент для LM Studio (чат + vision)
client = OpenAI(
    base_url=LMS_URL,
    api_key="lm-studio"   # LM Studio не проверяет ключ
)

print("✅ Клиент LM Studio инициализирован")

✅ Клиент LM Studio инициализирован


---
## Шаг 2: Вспомогательные функции

- `generate_image` — генерирует картинку через локальный **sd-server** (Krea-2)
- `gpt` — текстовый/мультимодальный запрос к **LM Studio**

In [4]:
def generate_image(prompt, width=SD_WIDTH, height=SD_HEIGHT, steps=SD_STEPS, cfg_scale=SD_CFG, seed=-1):
    """Генерация изображения через sd-server (Krea-2)"""
    payload = {
        "prompt": prompt,
        "negative_prompt": "",
        "steps": steps,
        "cfg_scale": cfg_scale,
        "width": width,
        "height": height,
        "seed": seed,
        "sampler_name": "euler",
        "batch_size": 1,
    }
    r = requests.post(f"{SD_URL}/sdapi/v1/txt2img", json=payload, timeout=600)
    r.raise_for_status()
    data = r.json()
    img_b64 = data["images"][0]
    image = Image.open(io.BytesIO(base64.b64decode(img_b64)))
    return image


def image_to_base64(image, fmt="JPEG"):
    buffered = io.BytesIO()
    if image.mode in ("RGBA", "P"):
        image = image.convert("RGB")
    image.save(buffered, format=fmt, quality=90)
    return base64.b64encode(buffered.getvalue()).decode()


def get_content(prompt, image=None):
    """Формирует content для chat.completions (текст или текст+картинка)"""
    if image is not None:
        image_b64 = image_to_base64(image)
        return [
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}
            }
        ]
    return prompt


def gpt(prompt, model=LMS_MODEL, image=None, temperature=0.7):
    """Запрос к LM Studio (текст или vision)"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": get_content(prompt, image)}],
        temperature=temperature,
        max_tokens=1024,
    )
    return response.choices[0].message.content

print("✅ Функции готовы")

✅ Функции готовы


---
## Шаг 3: Проверка — генерация промпта

In [ ]:
concept = "ВПИШИТЕ СВОЙ КОНЦЕПТ"

prompt = gpt(
    f"Придумай короткий, но выразительный промпт на английском для генерации изображения, "
    f"которое иллюстрирует концепцию «{concept}». "
    f"Только сам промпт, без пояснений и кавычек."
)
print(prompt)

Golden sunlight bursting through a laughing child's outstretched arms, vibrant confetti, warm bokeh, joyful cinematic light, expressive, high detail


## Проверка — генерация изображения

In [ ]:
img = generate_image(prompt)
display(img)
print(f"Размер: {img.size}")

---
## Шаг 4: Агентный цикл

1. LLM придумывает промпт по концепции  
2. Krea-2 генерирует изображение  
3. Vision-модель оценивает соответствие (0–1) и даёт рекомендации  
4. Если оценка ниже порога — улучшаем промпт и повторяем


In [7]:
def draw_concept(concept, max_iterations=5, threshold=0.9):
    current_prompt = None
    best_image = None
    best_score = -1.0
    history = []
    last_feedback = ""

    for i in range(1, max_iterations + 1):
        printx(f"### Итерация {i}/{max_iterations}")

        # 1. Создаём / улучшаем промпт
        if current_prompt is None:
            system_ask = (
                f"Придумай короткий выразительный промпт на английском для генерации изображения, "
                f"которое максимально точно иллюстрирует абстрактную концепцию «{concept}». "
                f"Только сам промпт, без пояснений, кавычек и преамбул."
            )
            current_prompt = gpt(system_ask).strip().strip('"').strip("'")
        else:
            improve_ask = (
                f"Концепция: «{concept}».\n"
                f"Текущий промпт: {current_prompt}\n"
                f"Предыдущая оценка и рекомендации:\n{last_feedback}\n\n"
                f"Улучши промпт (на английском), чтобы изображение лучше соответствовало концепции. "
                f"Верни только новый промпт, без пояснений."
            )
            current_prompt = gpt(improve_ask).strip().strip('"').strip("'")

        printx(f"**Промпт:** {current_prompt}")

        # 2. Генерируем изображение
        print("Генерирую изображение...")
        image = generate_image(current_prompt)
        display(image)

        # 3. Оцениваем через Vision
        eval_ask = (
            f"Оцени, насколько это изображение соответствует абстрактной концепции «{concept}».\n"
            f"Ответь СТРОГО в формате:\n"
            f"fit=<число от 0.0 до 1.0>\n"
            f"recommendations=<краткие рекомендации на русском, как улучшить>"
        )
        feedback = gpt(eval_ask, image=image)
        last_feedback = feedback
        printx(f"**Оценка модели:**\n```\n{feedback}\n```")

        # Парсим score
        score = 0.0
        for line in feedback.splitlines():
            line = line.strip().lower()
            if line.startswith("fit="):
                try:
                    score = float(line.split("=", 1)[1].strip().replace(",", "."))
                except Exception:
                    pass
                break

        history.append({
            "iteration": i,
            "prompt": current_prompt,
            "score": score,
            "feedback": feedback,
            "image": image
        })

        if score > best_score:
            best_score = score
            best_image = image

        printx(f"**Текущий fit:** {score:.2f} (лучший: {best_score:.2f})")

        if score >= threshold:
            printx(f"✅ Достигнут порог {threshold}. Останавливаемся.")
            break
    else:
        printx(f"Достигнуто максимальное число итераций ({max_iterations}).")

    printx(f"\n---\n## Лучший результат (fit={best_score:.2f})")
    if best_image:
        display(best_image)
        best_image.save(f"concept_{concept}_best.png")
        print(f"Сохранено: concept_{concept}_best.png")

    return best_image, best_score, history

### Запуск

In [ ]:
# Попробуйте: "счастье", "сложность", "гармония", "хаос", "филигранность", "время"
best_img, best_score, history = draw_concept("ВПИШИТЕ СВОЙ КОНЦЕПТ", max_iterations=4, threshold=0.92)

---
## Шаг 5: Эксперименты

In [ ]:
# draw_concept("сложность", max_iterations=3, threshold=0.9)
# draw_concept("гармония", max_iterations=3, threshold=0.9)

---
## Заключение

Агентный цикл работает полностью **локально**:

1. **LM Studio** (`qwen/qwen3.8-27b`) — генерирует/улучшает промпты + оценивает изображения (vision)
2. **sd-server + Krea-2** — рисует изображения

Возможные улучшения:
- Сохранять все промежуточные картинки
- Добавить память (историю всех промптов)
- Параллельно генерировать несколько вариантов
- Добавить параметр «стиль»
